In [25]:
# OpenAI usage — Knight Vision
# One cell, one run: each API call is a separate event (never merged).
# Chess: one explain call PER PLY (not one combined plan call).

from __future__ import annotations

import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from openai import OpenAI


def find_env_file() -> Path:
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / ".env"
        if candidate.is_file():
            return candidate
    return Path.cwd().resolve() / ".env"


def load_api_key(env_path: Path) -> str:
    load_dotenv(env_path, override=True)
    for name in ("OPENAI_API_KEY", "OPEN_AI_API_KEY"):
        value = os.environ.get(name, "").strip()
        if value:
            return value
    return ""


def mask_key(key: str) -> str:
    if len(key) < 12:
        return "(missing or too short)"
    return f"{key[:7]}…{key[-4:]} (len={len(key)})"


ENV_PATH = find_env_file()
API_KEY = load_api_key(ENV_PATH)

print("Env file:", ENV_PATH)
print("API key:", mask_key(API_KEY))
print("Configured:", bool(API_KEY))
if not API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in repo-root .env")

client = OpenAI(api_key=API_KEY)

PRICING = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 2.50, "output": 10.00},
}
DEFAULT_MODEL = "gpt-4o-mini"

# Each API call appends one row here — never batch multiple operations into one call.
events: list[dict] = []


def estimate_cost_usd(model: str, prompt_tokens: int, completion_tokens: int) -> float:
    p = PRICING.get(model, PRICING[DEFAULT_MODEL])
    return (prompt_tokens * p["input"] + completion_tokens * p["output"]) / 1_000_000


def ask_event(
    user_message: str,
    *,
    kind: str,
    label: str,
    model: str = DEFAULT_MODEL,
    system: str = "Reply in one short sentence.",
    max_tokens: int = 300,
    temperature: float = 0.4,
) -> str:
    """One OpenAI request = one event (one usage row, one printed block)."""
    t0 = datetime.now(timezone.utc)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_message},
        ],
    )
    text = (response.choices[0].message.content or "").strip()
    u = response.usage
    prompt_t = u.prompt_tokens if u else 0
    completion_t = u.completion_tokens if u else 0
    total_t = u.total_tokens if u else prompt_t + completion_t
    cost = estimate_cost_usd(model, prompt_t, completion_t)
    event_id = len(events) + 1
    events.append(
        {
            "event_id": event_id,
            "kind": kind,
            "label": label,
            "time_utc": t0.isoformat(timespec="seconds"),
            "model": model,
            "prompt_tokens": prompt_t,
            "completion_tokens": completion_t,
            "total_tokens": total_t,
            "cost_usd": round(cost, 6),
            "response": text,
        }
    )
    print(
        f"\n{'─' * 60}\n"
        f"EVENT {event_id} [{kind}] {label}\n"
        f"tokens: {prompt_t} prompt + {completion_t} completion = {total_t}  |  ${cost:.6f}\n"
        f"{'─' * 60}"
    )
    print(text)
    return text


def usage_df() -> pd.DataFrame:
    cols = [
        "event_id",
        "kind",
        "label",
        "time_utc",
        "model",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
        "cost_usd",
    ]
    if not events:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame(events)[cols]


def responses_df() -> pd.DataFrame:
    if not events:
        return pd.DataFrame(columns=["event_id", "kind", "label", "response"])
    return pd.DataFrame(events)["event_id", "kind", "label", "response"]


# ─── Event 1: smoke test (single unrelated call) ───
ask_event(
    "Say hello and confirm you received this.",
    kind="smoke_test",
    label="smoke test",
)

# ─── Chess: ONE API call per ply (do not combine plies into one prompt) ───
CHESS_CONTEXT = (
    "Student plays White (I). Opponent is Stockfish (you). "
    "Engine eval: +0.36 pawns for White. This is one move in the main line."
)
CHESS_SYSTEM = (
    "Explain ONLY the single chess move in the facts. 2-3 sentences. "
    "Student=I, opponent=you. Do not discuss other moves. Use only given facts."
)

CHESS_PLIES = [
    (1, "Plie 1. Student (I): Nf3 (UCI g1f3)."),
    (2, "Plie 2. Opponent (you): Nc6 (UCI b8c6)."),
    (3, "Plie 3. Student (I): Bb5 (UCI f1b5)."),
    (4, "Plie 4. Opponent (you): a6 (UCI a7a6)."),
    (
        5,
        "Plie 5. Student (I): Ba4 (UCI b5a4); gives check; "
        "after this move the side to move has 3 legal moves.",
    ),
]

print("\n" + "=" * 60)
print("CHESS EXPLAINS — one separate API call per ply")
print("=" * 60)

for ply_num, ply_facts in CHESS_PLIES:
    ask_event(
        f"{CHESS_CONTEXT}\n\nTHIS_MOVE_ONLY:\n{ply_facts}",
        kind="chess_explain_ply",
        label=f"chess explain ply {ply_num}",
        system=CHESS_SYSTEM,
        max_tokens=180,
        temperature=0.4,
    )

# ─── Usage: one row per event (no merging) ───
print("\n" + "=" * 60)
print("USAGE TABLE — one row per API call")
print("=" * 60)
display(usage_df())

print("\n" + "=" * 60)
print("RESPONSES — one block per event (see EVENT lines above too)")
print("=" * 60)
for e in events:
    print(f"\n### Event {e['event_id']}: [{e['kind']}] {e['label']}")
    print(e["response"])

df = usage_df()
if len(df):
    print(
        f"\nSession: {len(df)} separate API calls, "
        f"{df['total_tokens'].sum():,} tokens total, "
        f"${df['cost_usd'].sum():.6f} USD (estimated)"
    )
    chess_rows = df[df["kind"] == "chess_explain_ply"]
    if len(chess_rows):
        print(
            f"  Chess ply explains: {len(chess_rows)} calls, "
            f"{chess_rows['total_tokens'].sum():,} tokens, "
            f"${chess_rows['cost_usd'].sum():.6f}"
        )

out_usage = Path("openai_usage_log.csv")
usage_df().to_csv(out_usage, index=False)
out_responses = Path("openai_responses.csv")
responses_df().to_csv(out_responses, index=False)
print("\nWrote", out_usage.resolve())
print("Wrote", out_responses.resolve())


Env file: /Users/cwh/Data/Apps/KnightSchool/.env
API key: sk-proj…KJAA (len=164)
Configured: True

────────────────────────────────────────────────────────────
EVENT 1 [smoke_test] smoke test
tokens: 25 prompt + 9 completion = 34  |  $0.000009
────────────────────────────────────────────────────────────
Hello! I confirm I received your message.

CHESS EXPLAINS — one separate API call per ply

────────────────────────────────────────────────────────────
EVENT 2 [chess_explain_ply] chess explain ply 1
tokens: 105 prompt + 40 completion = 145  |  $0.000040
────────────────────────────────────────────────────────────
You played Nf3, developing your knight and controlling the center of the board. This move prepares for further development and helps secure the e5 square while also allowing for potential kingside castling.

────────────────────────────────────────────────────────────
EVENT 3 [chess_explain_ply] chess explain ply 2
tokens: 105 prompt + 43 completion = 148  |  $0.000042
───────

,event_id,kind,label,time_utc,model,prompt_tokens,completion_tokens,total_tokens,cost_usd
0,1,smoke_test,smoke test,2026-05-25T04:53:33+00:00,gpt-4o-mini,25,9,34,0.000009
1,2,chess_explain_ply,chess explain ply 1,2026-05-25T04:53:33+00:00,gpt-4o-mini,105,40,145,0.000040
2,3,chess_explain_ply,chess explain ply 2,2026-05-25T04:53:35+00:00,gpt-4o-mini,105,43,148,0.000042
3,4,chess_explain_ply,chess explain ply 3,2026-05-25T04:53:36+00:00,gpt-4o-mini,104,40,144,0.000040
4,5,chess_explain_ply,chess explain ply 4,2026-05-25T04:53:37+00:00,gpt-4o-mini,105,47,152,0.000044
5,6,chess_explain_ply,chess explain ply 5,2026-05-25T04:53:40+00:00,gpt-4o-mini,120,32,152,0.000037



RESPONSES — one block per event (see EVENT lines above too)

### Event 1: [smoke_test] smoke test
Hello! I confirm I received your message.

### Event 2: [chess_explain_ply] chess explain ply 1
You played Nf3, developing your knight and controlling the center of the board. This move prepares for further development and helps secure the e5 square while also allowing for potential kingside castling.

### Event 3: [chess_explain_ply] chess explain ply 2
In this move, you (Stockfish) play Nc6, developing your knight to a central square and increasing control over the board. This move also supports the pawn on e5 and prepares for potential future developments.

### Event 4: [chess_explain_ply] chess explain ply 3
You played Bb5, moving your bishop to b5. This puts pressure on the knight on c6 and potentially prepares for tactics involving the pinned knight if you decide to follow up with other moves.

### Event 5: [chess_explain_ply] chess explain ply 4
You played a6, which is a pawn move 

KeyError: ('event_id', 'kind', 'label', 'response')